In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

# Buat data mini
df1 = pd.DataFrame({
    'Restaurant': ['Cafe A', 'Cafe B', 'Cafe C'],
    'Rating': [4.5, 3.2, 3.9],
    'Value': [2, 4, 5]
})

# Atur biar angka desimalnya hanya 2 digit
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Siapkan data nya
df = pd.read_csv(r"B:\Python\DataSet\CSV\Zomato Restaurant Dataset.csv")
'''
['Restaurant ID',
 'Restaurant Name',
 'Country Code',
 'City',
 'Address',
 'Locality',
 'Locality Verbose',
 'Longitude',
 'Latitude',
 'Cuisines',
 'Average Cost for two',
 'Currency',
 'Has Table booking',
 'Has Online delivery',
 'Is delivering now',
 'Switch to order menu',
 'Price range',
 'Aggregate rating',
 'Rating color',
 'Rating text',
 'Votes']
'''

df.head()

1. Menghapus semua baris yang memiliki NaN dapat menimbulkan bias karena kemungkinan hubungan dengan variable lain dan interaksi antar nilai yang bisa mempengaruhi perhitungan statistik. Misalnya jika data yang hilang ternyata nilai rendah maka nilai data akan terlihat lebih tinggi dan sebaliknya.
2. Hilangnya suatu nilai tidak selalu acak melainkan juga bisa dipengaruhi oleh faktor yang berhungan. Sesuai namanya, MAR dapat dapat memanfaatkan variabel lain untuk imputasi karena 

In [ ]:
# Mencari data hilang
df.isna() #Menghasilkan boolean
df.notna() #Kebalikan isna

df.isna().sum() #Mengitung data hilang per kolom
df.isna().mean() * 100 #Menghitung persentase data hilang
df.isna().sum().sum() #Menghitung total data hilang di DataFrame
df.isna().any(axis=1).sum() #Menghitung jumlah baris yang setidaknya memiliki satu baris kosong

df.info()

In [ ]:
# Menggunakan fungsi
def missing_report(df):
    out = pd.DataFrame({
        'missing': df.isna().sum(),
        'percent': (df.isna().mean() * 100).round(2),
        'dtype': df.dtypes,
        'unique': df.nunique()
    })
    return out[out['missing'] > 0].sort_values('percent', ascending=False)

missing_report(df=df)

In [ ]:
# Viasualisasi data
sns.heatmap(df.isna(), cbar=False)
df["Aggregate rating"].describe() #Nilai min atau max di luar skala penilaian adalah redflag
df["Restaurant Name"].str.strip().eq('').sum() #Menghitung string kosong

# Mengubah placehoders menjadi data NaN sungguhan
df = df.replace(["N/A", "n/a", "NA", "-", "?", "unknown", ""], np.nan)
df.loc[(df["Aggregate rating"] < 0.0) | (df["Aggregate rating"] > 5.0)] = np.nan #Mengubah rating yang kurang dari 0 dan lebih dari 5 menjadi NaN
df["Restaurant Name"] = df['Restaurant Name'].str.strip().replace("", np.nan)

In [ ]:
# Missingness analist
df["MissingCuisines"] = df["Cuisines"].isna()

# Mengecek hubungan dengan kota
city_realativity = df.groupby('City')["MissingCuisines"].mean().reset_index()
filtered_city = city_realativity[city_realativity['MissingCuisines'] > 0]

#Mengecek hubungan dengan Price range
PriceRangeRelativity = df.groupby("Price range")['MissingCuisines'].mean().reset_index()
FilteredPriceRange = PriceRangeRelativity[PriceRangeRelativity["MissingCuisines"] > 0]

print(PriceRangeRelativity)
# Dan seterusnya

In [ ]:
# Menghapus (drop) data hilang
df.dropna() #Menghapus baris dengan minimal satu data hilang
df.dropna(how="all") #Menghapus baris jika semua data kosong
df.dropna(thresh=5) #Mempertahankan baris dengan setidaknya 5 nilai tak kosong
df.dropna(axis=1) #Menghapus kolom dengan setidaknya 1 nilai hilang
df.dropna(axis=1, thresh=(int(0.5 * len(df)))) #Membiarkan kolom yang terisi >+ 50%

1. `axis=0`(default) untuk baris dan `axis=1` untuk kolom
2. `how=any`(default) atau `how=all`
3. `thresh=n` jumlah minimum nilai tak kosong supaya kolom/baris tidak dihapus
4. `subset=[...]` membatasi pengecekan terhadap kolom spesifik


In [ ]:
# Mengisi data kosong

# fillna dasar
df['City'] = df['City'].fillna('Unknown') #Mengisi data kosong dnegan 'Unknown'
# Mengisi beberapa kolom sekaligus
dfX = df.fillna({
    'City': "Unknown",
    'Aggregate rating': np.nan
})

# Mengisi data kosong dengan hasil operasi statistik
df["Aggregate rating"] = df['Aggregate rating'].fillna(df["Aggregate rating"].mean()) #Data simetris
df['Price range'] = df['Price range'].fillna(df['Price range'].median()) #Untuk data skewed/outliers
df["City"] = df['City'].fillna(df['City'].mode()[0]) #Menggunakan data yang paling sering muncul

# Mengisi data kosong berdasarkan kolom yang berhubungan
df['Aggregate rating'] = df["Aggregate rating"].fillna(
    df.groupby("Rating color")["Aggregate rating"].transform('median')
)
